In [1]:
import pandas as pd
import numpy as np
import jenkspy

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

from pulp import (
    LpProblem,
    LpVariable,
    LpMaximize,
    lpSum,
    PULP_CBC_CMD,
    value,
)

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"
OUTPUT_CSV = Path("data") / "vulnerability.csv"

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# =============================================================================
# DEA CRS MULTIPLIER MODEL
# =============================================================================

def dea_crs(dmu_df, input_cols, output_cols):

    dmus = dmu_df.index.tolist()

    efficiencies = []

    for dmu in dmus:

        prob = LpProblem(
            f"DEA_{dmu}",
            LpMaximize
        )

        # OUTPUT WEIGHTS
        u = {
            col: LpVariable(
                f"u_{col}_{dmu}",
                lowBound=1e-6
            )
            for col in output_cols
        }

        # INPUT WEIGHTS
        v = {
            col: LpVariable(
                f"v_{col}_{dmu}",
                lowBound=1e-6
            )
            for col in input_cols
        }

        # OBJECTIVE
        prob += lpSum(
            u[r] * dmu_df.loc[dmu, r]
            for r in output_cols
        )

        # NORMALIZATION
        prob += (
            lpSum(
                v[i] * dmu_df.loc[dmu, i]
                for i in input_cols
            )
            == 1
        )

        # CRS CONSTRAINTS
        for j in dmus:

            prob += (
                lpSum(
                    u[r] * dmu_df.loc[j, r]
                    for r in output_cols
                )
                -
                lpSum(
                    v[i] * dmu_df.loc[j, i]
                    for i in input_cols
                )
                <= 0
            )

        prob.solve(
            PULP_CBC_CMD(msg=False)
        )

        eff = value(prob.objective)

        if eff is None:
            eff = np.nan
        else:
            eff = float(eff)
            eff = max(0.0, min(1.0, eff))

        efficiencies.append(eff)

    return efficiencies


# =============================================================================
# JENKS CLASSIFICATION
# =============================================================================

def assign_jenks_with_handling(data, n_classes=5):

    data = pd.Series(data)

    unique_vals = np.unique(data)

    if len(unique_vals) == 1:
        return pd.Series(
            [3] * len(data),
            index=data.index
        )

    if len(unique_vals) < n_classes:
        n_classes = len(unique_vals)

    while n_classes >= 2:

        try:

            breaks = jenkspy.jenks_breaks(
                data.values,
                n_classes=n_classes
            )

            unique_breaks = np.unique(breaks)

            if len(unique_breaks) < len(breaks):
                n_classes -= 1
                continue

            classes = pd.cut(
                data,
                bins=unique_breaks,
                labels=list(
                    range(1, n_classes + 1)
                ),
                include_lowest=True
            )

            return classes.astype(int)

        except Exception:
            n_classes -= 1

    return pd.Series(
        [3] * len(data),
        index=data.index
    )


# =============================================================================
# DISTRICT-MONTH AGGREGATION
# =============================================================================

district_df = (
    df.groupby(
        ["dtname", "timeperiod"],
        as_index=False
    )
    .agg(
        # ----------------------------------------------------------
        # SENSITIVITY
        # ----------------------------------------------------------
        aged_pop=("sum_aged_population", "sum"),
        young_pop=("sum_young_population", "sum"),
        no_sanitation=("rc_nosanitation_hhds_pct", "mean"),

        # PLFS
        workers_affected_pct=("workers-affected-pct", "mean"),

        # NFHS
        pct_ncd=("pct_ncd", "mean"),

        # ----------------------------------------------------------
        # COPING CAPACITY
        # ----------------------------------------------------------
        health_centers=("health_centres_count", "sum"),
        electricity=("avg_electricity", "mean"),
        piped_water=("rc_piped_hhds_pct", "mean")
    )
)

print(
    "District-month observations:",
    len(district_df)
)

print("\nDistrict DF columns:")
print(district_df.columns.tolist())

# =============================================================================
# MONTHWISE DEA
# =============================================================================

results = []

for month in sorted(
    district_df["timeperiod"].unique()
):

    print(f"Running DEA: {month}")

    month_df = (
        district_df[
            district_df["timeperiod"] == month
        ]
        .copy()
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # NORMALIZE
    # -------------------------------------------------------------------------

    scaler = MinMaxScaler()

    scale_cols = [
        "aged_pop",
        "young_pop",
        "no_sanitation",
        "workers_affected_pct",
        "pct_ncd",
        "health_centers",
        "electricity",
        "piped_water",
    ]

    month_df[scale_cols] = scaler.fit_transform(
        month_df[scale_cols]
    )

    # Avoid exact zeros
    month_df[scale_cols] += 1e-6

    # -------------------------------------------------------------------------
    # INVERT COPING CAPACITY
    # -------------------------------------------------------------------------

    month_df["inv_health_centers"] = (
        1 - month_df["health_centers"]
    )

    month_df["inv_electricity"] = (
        1 - month_df["electricity"]
    )

    month_df["inv_piped_water"] = (
        1 - month_df["piped_water"]
    )

    # -------------------------------------------------------------------------
    # CONSTANT OUTPUT
    # -------------------------------------------------------------------------

    month_df["constant_output"] = 1.0

    # -------------------------------------------------------------------------
    # DEA INPUTS
    # -------------------------------------------------------------------------

    INPUTS = [
        "aged_pop",
        "young_pop",
        "no_sanitation",
        "workers_affected_pct",
        "pct_ncd",
        "inv_health_centers",
        "inv_electricity",
        "inv_piped_water",
    ]

    OUTPUTS = [
        "constant_output"
    ]

    dea_df = month_df.copy()

    dea_df.index = dea_df["dtname"]

    dea_df["efficiency"] = dea_crs(
        dea_df,
        INPUTS,
        OUTPUTS
    )

    month_df["efficiency"] = (
        dea_df["efficiency"].values
    )

    # -------------------------------------------------------------------------
    # VULNERABILITY
    # -------------------------------------------------------------------------

    month_df["vulnerability_raw"] = (
        1 - month_df["efficiency"]
    )

    # -------------------------------------------------------------------------
    # JENKS NATURAL BREAKS
    # -------------------------------------------------------------------------

    month_df["vulnerability"] = (
        assign_jenks_with_handling(
            month_df["vulnerability_raw"],
            n_classes=5
        )
    )

    results.append(month_df)

# =============================================================================
# COMBINE
# =============================================================================

result_df = pd.concat(
    results,
    ignore_index=True
)

# =============================================================================
# OUTPUT
# =============================================================================

output_cols = [
    "dtname",
    "timeperiod",

    "aged_pop",
    "young_pop",
    "no_sanitation",
    "workers_affected_pct",
    "pct_ncd",

    "health_centers",
    "electricity",
    "piped_water",

    "inv_health_centers",
    "inv_electricity",
    "inv_piped_water",

    "efficiency",
    "vulnerability_raw",
    "vulnerability",
]

result_df[output_cols].to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:", OUTPUT_CSV)

print("\nEfficiency Summary")
print(
    result_df["efficiency"]
    .describe()
)

print("\nVulnerability Class Distribution")
print(
    result_df["vulnerability"]
    .value_counts()
    .sort_index()
)

print("\nMonthly Distribution")

for month in sorted(
    result_df["timeperiod"].unique()
):
    print(f"\n{month}")

    print(
        result_df[
            result_df["timeperiod"] == month
        ]["vulnerability"]
        .value_counts()
        .sort_index()
    )

print("\nPreview")
print(
    result_df.head()
)

Input shape: (11700, 30)

Columns:
['object_id', 'timeperiod', 'total_tender_awarded_value', 'heat-days-score', 'land-surface-temperature', 'year', 'sum_aged_population', 'sum_young_population', 'sum_population', 'health_centres_count', 'workers-affected-pct', 'net_sown_area_in_hac', 'avg_electricity', 'avg_tele', 'rc_piped_hhds_pct', 'rc_nosanitation_hhds_pct', 'total_hhd', 'revenue_ci', 'revenue_cr', 'HQ', 'are_new', 'women_sugar', 'men_sugar', 'women_bp', 'men_bp', 'pct_ncd', 'dtname', 'rc_area', 'land-surface-temperature-raster', 'district']
District-month observations: 2275

District DF columns:
['dtname', 'timeperiod', 'aged_pop', 'young_pop', 'no_sanitation', 'workers_affected_pct', 'pct_ncd', 'health_centers', 'electricity', 'piped_water']
Running DEA: 2021_01


/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/.venv/lib/python3.12/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


Running DEA: 2021_02


/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/.venv/lib/python3.12/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


Running DEA: 2021_03


...

Running DEA: 2026_04


/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/.venv/lib/python3.12/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


Running DEA: 2026_05


/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/.venv/lib/python3.12/site-packages/pulp/pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")



Saved: data/vulnerability.csv

Efficiency Summary
count    2275.000000
mean        0.939707
std         0.093988
min         0.676063
25%         0.892573
50%         1.000000
75%         1.000000
max         1.000000
Name: efficiency, dtype: float64

Vulnerability Class Distribution
vulnerability
1    1430
2     195
3     390
4     195
5      65
Name: count, dtype: int64

Monthly Distribution

2021_01
vulnerability
1    22
2     3
3     6
4     3
5     1
Name: count, dtype: int64

2021_02
vulnerability
1    22
2     3
3     6
4     3
5     1
Name: count, dtype: int64

2021_03
vulnerability
1    22
2     3
3     6
4     3
5     1
Name: count, dtype: int64

2021_04
vulnerability
1    22
2     3
3     6
4     3
5     1
Name: count, dtype: int64

2021_05
vulnerability
1    22
2     3
3     6
4     3
5     1
Name: count, dtype: int64

2021_06
vulnerability
1    22
2     3
3     6
4     3
5     1
Name: count, dtype: int64

2021_07
vulnerability
1    22
2     3
3     6
4     3
5     1
Name: